In [1]:
import sys
import math
import numpy as np


In [2]:
#TODO don't use cosine_similarity because it does not distinguish (10,10) and (1,1) !!!


In [9]:
INT_NONE = -sys.maxsize - 1


def cosine_similarity(a,b):
    return np.dot(a, b)/(np.linalg.norm(a) * np.linalg.norm(b))


def max_corner_distance(dims):
    """
    Compute the maximum Euclidean distance (corner-to-corner) for a space
    where each dimension i is bounded in [0, M_i].
    Parameters:
        dims : array-like
            Maximum values for each dimension.
    Returns:
        float : Maximum Euclidean distance.
    """
    #return np.sqrt(np.sum(np.array(dims)**2))
    return np.linalg.norm(dims)

assert(round(max_corner_distance(([10,10])))==14)
# state = (previous_action,)+(1 if reward > 0 else 0,1 if reward < 0 else 0)+(racket_col, ball_col)
assert(round(math.ceil(max_corner_distance([3,1,1,178,178])))==252)


def max_corner_distance_min_max(minmax):
    """
    Compute the maximum Euclidean distance (corner-to-corner) for a space
    where each dimension i is bounded in [Min_i, Max_i].
    Parameters:
        minmax : array-like
            Min and Max values (lists) for each dimension.
    Returns:
        float : Maximum Euclidean distance.
    """
    squared_sum = 0.0
    for mi, ma in minmax:
        assert(mi < ma)
        diff = ma - mi
        squared_sum += diff * diff
    #TODO use np.linalg.norm for sqrt(sum(x**2))
    return np.sqrt(squared_sum)

assert(round(max_corner_distance_min_max(([0,10],[0,10])))==14)
assert(round(max_corner_distance_min_max(([-10,10],[-10,10])))==28)
    

def norm_euclidean_distance(a,b,dims,dist_max):
    """
    Euclidean distance between two vectors noralized by the longest vector, assuming both vectores are in the same  
    """
    #d = np.linalg.norm(np.array(a) - np.array(b)) / dist_max
    c = [(ai-bi if (ai !=INT_NONE and bi != INT_NONE) else di) for ai, bi, di in zip(a,b,dims)]
    if dist_max is None:
        dist_max = max_corner_distance(dims)
    d = np.linalg.norm(c) / dist_max
    if not (d >= 0 and d <=1.0):
        print(a)
        print(b)
        print(c)
        print(d,np.linalg.norm(c),dist_max)
    assert(d >= 0 and d <=1.0) # to make sure it is in range
    return d


def norm_similarity(a,b,method='1/(1+d)',dims=None,dist_max=None):
    if method == 'cos':
        return cosine_similarity(a,b) 
    if method == '1-dnorm': # equivalent to '1-d/max'
        a = [(ai / di if ai !=INT_NONE else ai) for ai, di in zip(a,dims)]
        b = [(bi / di if bi !=INT_NONE else bi) for bi, di in zip(b,dims)]
        c = [(ai-bi if (ai !=INT_NONE and bi != INT_NONE) else 1) for ai, bi in zip(a,b)]
        d = np.linalg.norm(c) / math.sqrt(len(dims)) # sqrt(sum(1**2))
        if not (d >= 0 and d <=1.0):
            print(a)
            print(b)
            print(c)
            print(d)
        assert(d >= 0 and d <=1.0) # to make sure it is in range
        return 1 - d
    if method == '1-d/max': 
        #return max(0.0, 1.0 - np.linalg.norm(np.array(a) - np.array(b)) / dist_max)
        return 1 - norm_euclidean_distance(a,b,dims,dist_max) # do this instead of just d for interrnal assertion purposes 
    d = np.linalg.norm(np.array(a) - np.array(b))
    if method == 'exp(-d)':
        return np.exp(-d)
    else: #if method == '1/(1+d)':
        #print('---',d,1.0 / (1.0 + d))
        return 1.0 / (1.0 + d)


# 'cos': The worst - this illustrates that cosine similarity is not usable completely
# it does not distinguish (10,10) and (1,1)
assert(float(round(norm_similarity((10,10),(10,10),'cos'),2))==1.0)
assert(float(round(norm_similarity((10,10),(10,9 ),'cos'),2))==1.0)
assert(float(round(norm_similarity((10,10),(10,5 ),'cos'),2))==0.95)
assert(float(round(norm_similarity((10,10),( 9,9 ),'cos'),2))==1.0) # problem!
assert(float(round(norm_similarity((10,10),(10,0 ),'cos'),2))==0.71)
#assert(np.isnan(float(round(norm_similarity((10,10),( 0,0 ),'cos'),2)))) # problem (commented out to suppress warning)!
assert(float(round(norm_similarity((10,),(1,),'cos'),2))==1.0) # problem!
assert(float(round(norm_similarity((10,10),(1,1),'cos'),2)==1.0)) # problem!
assert(float(round(norm_similarity((10,10,0,0,0),(1,1,0,0,0),'cos'),2))==1.0) # problem!
assert(float(round(norm_similarity((10,10,5,5,5),(1,1,5,5,5),'cos'),2))==0.65)

# '1/(1+d)': Not than good?
assert(float(round(norm_similarity((10,10),(10,10)),2))==1.0)
assert(float(round(norm_similarity((10,10),(10,9)),2))==0.5) # not good...
assert(float(round(norm_similarity((10,10),(10,5)),2))==0.17)
assert(float(round(norm_similarity((10,10),( 9,9 )),2))==0.41)
assert(float(round(norm_similarity((10,10),(10,0 )),2))==0.09)
assert(float(round(norm_similarity((10,10),( 0,0 )),2))==0.07)
assert(float(round(norm_similarity((10,),(1,)),2))==0.1)
assert(float(round(norm_similarity((10,10),(1,1)),2))==0.07)
assert(float(round(norm_similarity((10,10,0,0,0),(1,1,0,0,0)),2))==0.07)
assert(float(round(norm_similarity((10,10,5,5,5),(1,1,5,5,5)),2))==0.07)
assert(round(norm_similarity((10,10,5,5,5),(10,10,5,5,5)),2)==1.0)
assert(round(norm_similarity((10,10,5,5,5),(10,10,5,5,4)),2)==0.5) # not good...

# '1-d/max': The best!?
assert(float(round(norm_similarity((10,10),(10,10),'1-d/max',(10,10)),2))==1.0)
assert(float(round(norm_similarity((10,10),(10,9),'1-d/max',(10,10)),2))==0.93)
assert(float(round(norm_similarity((10,10),(10,5),'1-d/max',(10,10)),2))==0.65)
assert(round(norm_similarity((10,10),( 9,9 ),'1-d/max',(10,10)),2)==0.9)
assert(round(norm_similarity((10,10),(10,0 ),'1-d/max',(10,10)),2)==0.29)
assert(round(norm_similarity((10,10),( 0,0 ),'1-d/max',(10,10)),2)==0.0)
assert(round(norm_similarity((10,),(1,),'1-d/max',(10,)),2)==0.1)
assert(float(round(norm_similarity((10,10),(1,1),'1-d/max',(10,10)),2))==0.1)
assert(round(norm_similarity((10,10,0,0,0),(1,1,0,0,0),'1-d/max',   (10,10,10,10,10)),2)==0.43)
assert(round(norm_similarity((10,10,10,10,10),(1,1,1,1,1),'1-d/max',(10,10,10,10,10)),2)==0.1)
assert(round(norm_similarity((10,10,5,5,5),(10,10,5,5,5),'1-d/max', (10,10,10,10,10)),2)==1.0)
assert(round(norm_similarity((10,10,5,5,5),(1,1,5,5,5),'1-d/max',   (10,10,10,10,10)),2)==0.43)
assert(round(norm_similarity((10,10,5,5,5),(10,10,5,5,5),'1-d/max', (10,10,10,10,10)),2)==1.0)
assert(round(norm_similarity((10,10,5,5,5),(10,10,5,5,4),'1-d/max', (10,10,10,10,10)),2)==0.96) # good!
assert(round(norm_similarity([10,10,10],[10,10,10],'1-d/max',[10,10,10]),2)==1.0)
assert(round(norm_similarity([10,10,INT_NONE],[10,10,10],'1-d/max',[10,10,10]),2)==0.42)
assert(round(norm_similarity([10,INT_NONE,INT_NONE],[10,10,10],'1-d/max',[10,10,10]),2)==0.18)


def dimensions_init(N):
    return tuple([None] * 2 for _ in range(N))

def measure_dimensions(min_max,state):
    for i, var in enumerate(state):
        mm = min_max[i]
        if mm[0] is None or mm[0] > var: # min
            mm[0] = var
        if mm[1] is None or mm[1] < var: # max
            mm[1] = var

_space_min_max = dimensions_init(2)
measure_dimensions(_space_min_max,(10,10))
measure_dimensions(_space_min_max,(1,100))
assert(str(_space_min_max)=="([1, 10], [10, 100])")


In [4]:
for t in [
((10,10),(10,10),(10,10),1.0),
((10,10),(10,9), (10,10),0.93),
((10,10),(10,5), (10,10),0.65),
((10,10),( 9,9 ),(10,10),0.9),
((10,10),(10,0 ),(10,10),0.29),
((10,10),( 0,0 ),(10,10),0.0),
((10,),(1,),     (10,),  0.1),
((10,10),(1,1),  (10,10),0.1),
((10,10,0,0,0),(1,1,0,0,0),   (10,10,10,10,10),0.43),
((10,10,10,10,10),(1,1,1,1,1),(10,10,10,10,10),0.1),
((10,10,5,5,5),(10,10,5,5,5), (10,10,10,10,10),1.0),
((10,10,5,5,5),(1,1,5,5,5),   (10,10,10,10,10),0.43),
((10,10,5,5,5),(10,10,5,5,5), (10,10,10,10,10),1.0),
((10,10,5,5,5),(10,10,5,5,4), (10,10,10,10,10),0.96),
((10,10,10),(10,10,10),       (10,10,10),      1.0),
((10,10,INT_NONE),(10,10,10), (10,10,10),      0.42),
((10,INT_NONE,INT_NONE),(10,10,10),(10,10,10), 0.18)]:
    #print(round(norm_similarity(t[0],t[1],'1-d/max',t[2]),2),t[3])
    assert(round(norm_similarity(t[0],t[1],'1-d/max',t[2]),2)==t[3])


In [5]:
for t in [
((10,10),(10,10),(10,10),1.0),
((10,10),(10,9), (10,10),0.93),
((10,10),(10,5), (10,10),0.65),
((10,10),( 9,9 ),(10,10),0.9),
((10,10),(10,0 ),(10,10),0.29),
((10,10),( 0,0 ),(10,10),0.0),
((10,),(1,),     (10,),  0.1),
((10,10),(1,1),  (10,10),0.1),
((10,10,0,0,0),(1,1,0,0,0),   (10,10,10,10,10),0.43),
((10,10,10,10,10),(1,1,1,1,1),(10,10,10,10,10),0.1),
((10,10,5,5,5),(10,10,5,5,5), (10,10,10,10,10),1.0),
((10,10,5,5,5),(1,1,5,5,5),   (10,10,10,10,10),0.43),
((10,10,5,5,5),(10,10,5,5,5), (10,10,10,10,10),1.0),
((10,10,5,5,5),(10,10,5,5,4), (10,10,10,10,10),0.96),
((10,10,10),(10,10,10),       (10,10,10),      1.0),
((10,10,INT_NONE),(10,10,10), (10,10,10),      0.42),
((10,INT_NONE,INT_NONE),(10,10,10),(10,10,10), 0.18)]:
    #print(round(norm_similarity(t[0],t[1],'1-dnorm',t[2]),2),t[3])
    assert(round(norm_similarity(t[0],t[1],'1-dnorm',t[2]),2)==t[3])

In [6]:
max_corner_distance([1]),max_corner_distance([1,1]),max_corner_distance([1,1,1])

(np.float64(1.0),
 np.float64(1.4142135623730951),
 np.float64(1.7320508075688772))

In [7]:
np.linalg.norm([1]),np.linalg.norm([1,1]),np.linalg.norm([1,1,1])

(np.float64(1.0),
 np.float64(1.4142135623730951),
 np.float64(1.7320508075688772))

In [8]:
math.sqrt(1), math.sqrt(2), math.sqrt(3)  

(1.0, 1.4142135623730951, 1.7320508075688772)